# 05 - Fine-tuned encoders + Focal / nnPU
Smoke test first: 64 rows, 1 epoch, peak memory vs 16 GB (T4), NaN check for DeBERTa in fp16.

In [ ]:
# Colab setup: repository at /content/kev-text (git clone or unzip), then run all.
import os, sys
REPO = "/content/kev-text"
if os.path.exists(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
# !pip install -q -r requirements.txt          # once per Colab session
import numpy as np, pandas as pd
from src import config, io_utils, labels, evaluate
SYN = False   # True = dry run on src.synthetic; everything goes to outputs_synthetic/

In [ ]:
if SYN:
    from src import synthetic
    df = synthetic.make(6000)
    if not config.SPLITS_LOCK.exists():                   # mirrors the extras notebook 02 writes
        from src import sensitivity
        sp0 = labels.assign_split(df["published_date"])
        m = sensitivity.kev_retroactive_monthly(df[["cve_id", "kev_date_added"]].dropna(),
                                                df[["cve_id", "published_date"]])
        vb = df[(sp0 == "val") & (df["published_date"] > config.VAL_A_END)]
        labels.write_lock(df, "initial_seal", "synthetic dry run",
                          extra={"s1": sensitivity.s1_record(m, df, sp0),
                                 "conformal_alpha": config.conformal_alpha(int(vb["label_12m"].sum()))})
else:
    df = pd.read_parquet(config.DATA / "dataset_labelled.parquet")
lock = labels.require_lock()
X, y, split = labels.get_xy(df, labels.CLEAN)
split.value_counts()

In [ ]:
from src import models
SMOKE = True
smoke_minutes = {}
for key, name in config.ENCODERS.items():
    with evaluate.log_run(key, 0, "smoke", is_synthetic=SYN, smoke=True) as rec:
        r = models.finetune_encoder(X, y, split, name, seed=0, smoke=True, fp16=(key != "deberta"))
        rec["max_memory_gb"] = r["max_memory_gb"]
    smoke_minutes[key] = rec["seconds"] / 60
smoke_minutes      # -> evaluate.run_count_table(smoke_minutes=..., n_train=(split == "train").sum())

In [ ]:
if not SMOKE:
    for key, name in config.ENCODERS.items():
        for seed in config.SEEDS_FINAL:
            with evaluate.log_run(key, seed, "----", is_synthetic=SYN):
                r = models.finetune_encoder(X, y, split, name, seed=seed)
            evaluate.save_scores(key, seed, r["scores"], is_synthetic=SYN)
        evaluate.lock_from_val_a(key, y, {"lr": 2e-5, "epochs": 3}, is_synthetic=SYN)
    best = evaluate.lock_best_encoder(list(config.ENCODERS))
    print("best encoder on val_a:", best)

In [ ]:
if not SMOKE:
    best = io_utils.read_json(config.SELECTION_LOCK)["best_encoder"]
    for loss in ("focal", "nnpu"):
        for seed in config.SEEDS_FINAL:
            with evaluate.log_run(f"{best}-{loss}", seed, "----", is_synthetic=SYN):
                r = models.finetune_encoder(X, y, split, config.ENCODERS[best], seed=seed, loss=loss)
            evaluate.save_scores(f"{best}-{loss}", seed, r["scores"], is_synthetic=SYN)
        evaluate.lock_from_val_a(f"{best}-{loss}", y, {"loss": loss}, is_synthetic=SYN)